# Phase 8 Wave 2 External GBDT Comparison

This notebook orchestrates the authorized Wave 2 comparison by running the
model code inside the separate Wave 2 environment created by notebook 08b.
It does not install packages, mutate `.venv`, write to the main experiment
log, create submissions, use leaderboard feedback, or open Phase 9/10/11.


In [1]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

AUTHORIZED_HEAD = "98d8bb9bc0cc2cccd0c3722a9efebf56ab63021e"
EXPERIMENT_ID = "phase8_wave2_external_gbdt_v1"
os.environ["MPLBACKEND"] = "Agg"
REPO_ROOT = Path(r"C:\GitHub\reto_Tokio")
assert (REPO_ROOT / ".git").exists(), f"Repository marker not found: {REPO_ROOT / '.git'}"
ENV_DIR = Path(r"C:\tmp\reto_tokio_phase8_wave2_env")
ENV_PYTHON = ENV_DIR / "Scripts" / "python.exe"
TEMP_RUNNER = Path(r"C:\tmp\phase8_wave2_external_gbdt_v1_runner.py")
FORBIDDEN_PATHS = [
    "data/input",
    "notebooks/_official",
    "references",
    "outputs/submissions",
    "logs/experiment_log.csv",
    ".vscode/settings.json",
]

def run_cmd(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"
    result = subprocess.run(args, cwd=str(REPO_ROOT), text=True, capture_output=True, env=env)
    print("$", " ".join(args))
    if result.stdout.strip():
        print(result.stdout.strip()[-5000:])
    if result.stderr.strip():
        print(result.stderr.strip()[-5000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with code {result.returncode}: {' '.join(args)}")
    return result

head = run_cmd(["git", "rev-parse", "HEAD"]).stdout.strip()
assert head == AUTHORIZED_HEAD, f"HEAD mismatch: {head}"
assert not run_cmd(["git", "diff", "--cached", "--name-only"]).stdout.strip(), "Staged files present"
run_cmd(["git", "diff", "--check"])
assert not run_cmd(["git", "diff", "--name-only", "--", *FORBIDDEN_PATHS]).stdout.strip(), "Forbidden-path diff present"
assert ENV_PYTHON.exists(), f"Separate Wave 2 Python is missing: {ENV_PYTHON}"

blocked_outputs = [
    REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_model_summary.csv",
    REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_fold_metrics.csv",
    REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_slice_report.csv",
    REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_validation_report.md",
    REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_experiment_log_candidate.csv",
    REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_artifact_manifest.csv",
    *[REPO_ROOT / "outputs" / "oof" / f"{EXPERIMENT_ID}_{model_key}_oof_predictions.csv" for model_key in ["xgboost", "lightgbm", "catboost"]],
]
collisions = [str(path) for path in blocked_outputs if path.exists()]
if collisions:
    raise FileExistsError("Wave 2 artifact collision(s): " + "; ".join(collisions))

TEMP_RUNNER.write_text('\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport importlib\nimport json\nimport math\nimport os\nimport platform\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder\n\ntry:\n    from xgboost import XGBClassifier\n    XGBOOST_IMPORT_ERROR = ""\nexcept Exception as exc:\n    XGBClassifier = None\n    XGBOOST_IMPORT_ERROR = repr(exc)\n\ntry:\n    from lightgbm import LGBMClassifier\n    LIGHTGBM_IMPORT_ERROR = ""\nexcept Exception as exc:\n    LGBMClassifier = None\n    LIGHTGBM_IMPORT_ERROR = repr(exc)\n\ntry:\n    from catboost import CatBoostClassifier\n    CATBOOST_IMPORT_ERROR = ""\nexcept Exception as exc:\n    CatBoostClassifier = None\n    CATBOOST_IMPORT_ERROR = repr(exc)\n\nos.environ["MPLBACKEND"] = "Agg"\n\nAUTHORIZED_HEAD = "98d8bb9bc0cc2cccd0c3722a9efebf56ab63021e"\nEXPERIMENT_ID = "phase8_wave2_external_gbdt_v1"\nPROJECT_SEED = 42\nOOF_GAIN_THRESHOLD = 0.005436\nSLICE_MIN_N = 50\nSLICE_DEGRADATION_THRESHOLD = -0.02\nFOLD_SHA16 = "96937649526bcadb"\nM0_EXPECTED_AUC = 0.8116502602456482\nM0_TOL = 1e-9\nCATBOOST_2B_AUTHORIZED = True\nCATBOOST_SCHOOL_EXCLUSION_RECONFIRMED = True\n\nREPO_ROOT = Path(r"C:\\GitHub\\reto_Tokio")\nif not (REPO_ROOT / ".git").exists():\n    raise RuntimeError(f"Repository marker not found: {REPO_ROOT / \'.git\'}")\nTRAIN_PATH = REPO_ROOT / "data" / "input" / "train.csv"\nFOLD_PATH = REPO_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"\nM0_OOF_PATH = REPO_ROOT / "outputs" / "oof" / "phase8_model_family_comparison_v1_m0_random_forest_frozen_oof_predictions.csv"\nM1_OOF_PATH = REPO_ROOT / "outputs" / "oof" / "phase8_model_family_comparison_v1_m1_logistic_regression_oof_predictions.csv"\nMAIN_LOG_PATH = REPO_ROOT / "logs" / "experiment_log.csv"\nDEPENDENCY_REPORT = REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_dependency_report.csv"\nENVIRONMENT_REPORT = REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_environment_report.md"\n\nMODEL_SUMMARY_PATH = REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_model_summary.csv"\nFOLD_METRICS_PATH = REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_fold_metrics.csv"\nSLICE_REPORT_PATH = REPO_ROOT / "outputs" / "validation" / f"{EXPERIMENT_ID}_slice_report.csv"\nVALIDATION_REPORT_PATH = REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_validation_report.md"\nCANDIDATE_LOG_PATH = REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_experiment_log_candidate.csv"\nARTIFACT_MANIFEST_PATH = REPO_ROOT / "outputs" / "reports" / f"{EXPERIMENT_ID}_artifact_manifest.csv"\n\nBASE_FEATURES = [\n    "Year", "Age", "Height", "Weight", "Sprint_40yd", "Vertical_Jump",\n    "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle",\n    "Player_Type", "Position_Type", "Position",\n]\nPHYSICAL_TEST_COLUMNS = [\n    "Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps",\n    "Broad_Jump", "Agility_3cone", "Shuttle",\n]\nMISSINGNESS_FLAG_COLUMNS = [\n    "Age_missing", "Sprint_40yd_missing", "Vertical_Jump_missing",\n    "Bench_Press_Reps_missing", "Broad_Jump_missing", "Agility_3cone_missing",\n    "Shuttle_missing",\n]\nF2_FEATURES = BASE_FEATURES + MISSINGNESS_FLAG_COLUMNS + ["available_measurement_count"]\nNUMERIC_FEATURES = [\n    "Year", "Age", "Height", "Weight", "Sprint_40yd", "Vertical_Jump",\n    "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle",\n    *MISSINGNESS_FLAG_COLUMNS, "available_measurement_count",\n]\nCATEGORICAL_FEATURES = ["Player_Type", "Position_Type", "Position"]\nKNOWN_MISSING_COUNTS = {\n    "Age": 435,\n    "Sprint_40yd": 145,\n    "Vertical_Jump": 554,\n    "Bench_Press_Reps": 721,\n    "Broad_Jump": 581,\n    "Agility_3cone": 970,\n    "Shuttle": 912,\n}\nFORBIDDEN_PATHS = [\n    "data/input",\n    "notebooks/_official",\n    "references",\n    "outputs/submissions",\n    "logs/experiment_log.csv",\n    ".vscode/settings.json",\n]\n\n\ndef run_cmd(args: list[str], check: bool = True) -> subprocess.CompletedProcess:\n    env = os.environ.copy()\n    env["MPLBACKEND"] = "Agg"\n    result = subprocess.run(args, cwd=str(REPO_ROOT), text=True, capture_output=True, env=env)\n    if check and result.returncode != 0:\n        raise RuntimeError(f"Command failed {args}: {result.stderr}")\n    return result\n\n\ndef sha256_bytes(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef safe_write_csv(path: Path, frame: pd.DataFrame) -> None:\n    if path.exists():\n        raise FileExistsError(f"Refusing to overwrite artifact: {path}")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(path, index=False)\n\n\ndef safe_write_text(path: Path, text: str) -> None:\n    if path.exists():\n        raise FileExistsError(f"Refusing to overwrite artifact: {path}")\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(text, encoding="utf-8")\n\n\ndef positive_class_proba(estimator, matrix) -> np.ndarray:\n    classes = np.asarray(getattr(estimator, "classes_", []))\n    matches = np.where(classes == 1)[0]\n    if len(matches) != 1:\n        raise RuntimeError(f"Could not locate positive class label 1 exactly once; classes_={classes!r}")\n    proba = estimator.predict_proba(matrix)[:, int(matches[0])]\n    proba = np.asarray(proba, dtype=float)\n    if not np.isfinite(proba).all():\n        raise RuntimeError("Predicted probabilities contain NaN or infinity")\n    if ((proba < 0.0) | (proba > 1.0)).any():\n        raise RuntimeError("Predicted probabilities outside [0, 1]")\n    return proba\n\n\ndef auc_safe(y_true: np.ndarray, proba: np.ndarray) -> float:\n    if len(np.unique(y_true)) < 2:\n        return float("nan")\n    return float(roc_auc_score(y_true, proba))\n\n\ndef make_preprocessor() -> ColumnTransformer:\n    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])\n    categorical_pipe = Pipeline([\n        ("imputer", SimpleImputer(strategy="most_frequent")),\n        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),\n    ])\n    return ColumnTransformer(\n        transformers=[\n            ("num", numeric_pipe, NUMERIC_FEATURES),\n            ("cat", categorical_pipe, CATEGORICAL_FEATURES),\n        ],\n        remainder="drop",\n        verbose_feature_names_out=True,\n    )\n\n\ndef add_f2_features(train: pd.DataFrame) -> pd.DataFrame:\n    x = train.copy()\n    for col, expected_missing in KNOWN_MISSING_COUNTS.items():\n        if col not in x.columns:\n            raise RuntimeError(f"Missing required F2 source column: {col}")\n        observed = int(x[col].isna().sum())\n        if observed != expected_missing:\n            raise RuntimeError(f"Missing count mismatch for {col}: {observed} != {expected_missing}")\n    x["Age_missing"] = x["Age"].isna().astype(int)\n    for col in PHYSICAL_TEST_COLUMNS:\n        x[f"{col}_missing"] = x[col].isna().astype(int)\n    x["available_measurement_count"] = x[PHYSICAL_TEST_COLUMNS].notna().sum(axis=1).astype(int)\n    features = x[F2_FEATURES].copy()\n    forbidden = {"Id", "Drafted", "School"} & set(features.columns)\n    if forbidden:\n        raise RuntimeError(f"Forbidden columns in feature matrix: {sorted(forbidden)}")\n    if features.shape[1] != 21:\n        raise RuntimeError(f"F2 feature count mismatch: {features.shape[1]} != 21")\n    return features\n\n\ndef diagnostic_frame(train: pd.DataFrame, features: pd.DataFrame) -> pd.DataFrame:\n    diag = pd.DataFrame(index=train.index)\n    for col in ["Player_Type", "Position_Type", "Year"]:\n        diag[col] = train[col].astype(str)\n    diag["Age_missing"] = features["Age_missing"].map({0: "0", 1: "1"})\n    diag["available_measurement_count"] = features["available_measurement_count"].astype(str)\n\n    def completeness_group(count: int) -> str:\n        if count == 0:\n            return "none"\n        if count <= 3:\n            return "low"\n        if count <= 5:\n            return "partial"\n        return "complete"\n\n    diag["measurement_completeness_group"] = features["available_measurement_count"].map(completeness_group)\n    school_counts = train["School"].value_counts(dropna=False)\n    diag["frequent_vs_rare_school_group"] = train["School"].map(lambda school: "rare" if school_counts.loc[school] < 5 else "frequent")\n    return diag\n\n\ndef validate_oof_frame(frame: pd.DataFrame, folds: pd.DataFrame, name: str) -> None:\n    required = {"Id", "fold", "y_true", "y_pred_proba"}\n    if not required.issubset(frame.columns):\n        raise RuntimeError(f"{name} missing columns: {sorted(required - set(frame.columns))}")\n    if len(frame) != 2781:\n        raise RuntimeError(f"{name} row count mismatch: {len(frame)}")\n    if not frame["Id"].equals(folds["Id"]):\n        raise RuntimeError(f"{name} Id order mismatch vs frozen folds")\n    if not frame["fold"].equals(folds["fold"]):\n        raise RuntimeError(f"{name} fold alignment mismatch")\n    proba = frame["y_pred_proba"].to_numpy(dtype=float)\n    if not np.isfinite(proba).all() or ((proba < 0.0) | (proba > 1.0)).any():\n        raise RuntimeError(f"{name} invalid probabilities")\n    for fold in sorted(folds["fold"].unique()):\n        y_fold = frame.loc[frame["fold"] == fold, "y_true"].to_numpy()\n        if len(np.unique(y_fold)) < 2:\n            raise RuntimeError(f"{name} fold {fold} has one class")\n\n\ndef fold_auc_table(name: str, oof: pd.DataFrame) -> dict[int, float]:\n    return {\n        int(fold): auc_safe(group["y_true"].to_numpy(), group["y_pred_proba"].to_numpy())\n        for fold, group in oof.groupby("fold", sort=True)\n    }\n\n\ndef classification_for(delta: float, same_sign: int, slice_guard: bool, failed: bool = False) -> str:\n    if failed:\n        return "failed_run"\n    if delta >= OOF_GAIN_THRESHOLD and same_sign >= 4:\n        return "escalated" if slice_guard else "promotable_evidence"\n    return "no_qualifying_evidence"\n\n\ndef markdown_table(frame: pd.DataFrame) -> str:\n    if frame.empty:\n        return "No rows."\n    columns = [str(col) for col in frame.columns]\n    rows = []\n    for _, row in frame.iterrows():\n        rows.append([str(row[col]) for col in frame.columns])\n    def esc(value: str) -> str:\n        return value.replace("|", "\\\\|").replace("\\n", " ")\n    lines = [\n        "| " + " | ".join(esc(col) for col in columns) + " |",\n        "| " + " | ".join("---" for _ in columns) + " |",\n    ]\n    for row in rows:\n        lines.append("| " + " | ".join(esc(value) for value in row) + " |")\n    return "\\n".join(lines)\n\n\nhead = run_cmd(["git", "rev-parse", "HEAD"]).stdout.strip()\nif head != AUTHORIZED_HEAD:\n    raise RuntimeError(f"HEAD mismatch: {head}")\nif run_cmd(["git", "diff", "--cached", "--name-only"]).stdout.strip():\n    raise RuntimeError("Staged files present")\nrun_cmd(["git", "diff", "--check"])\nif run_cmd(["git", "diff", "--name-only", "--", *FORBIDDEN_PATHS]).stdout.strip():\n    raise RuntimeError("Forbidden-path diff present")\n\nlog_before = sha256_bytes(MAIN_LOG_PATH)\nfor artifact in [\n    MODEL_SUMMARY_PATH,\n    FOLD_METRICS_PATH,\n    SLICE_REPORT_PATH,\n    VALIDATION_REPORT_PATH,\n    CANDIDATE_LOG_PATH,\n    ARTIFACT_MANIFEST_PATH,\n]:\n    if artifact.exists():\n        raise FileExistsError(f"Wave 2 artifact already exists: {artifact}")\n\nfor model_key in ["xgboost", "lightgbm", "catboost"]:\n    path = REPO_ROOT / "outputs" / "oof" / f"{EXPERIMENT_ID}_{model_key}_oof_predictions.csv"\n    if path.exists():\n        raise FileExistsError(f"Wave 2 OOF artifact already exists: {path}")\n\ntrain = pd.read_csv(TRAIN_PATH)\nif train.shape != (2781, 16):\n    raise RuntimeError(f"Unexpected train shape: {train.shape}")\nif "School" not in train.columns:\n    raise RuntimeError("School column missing for diagnostic-only slice")\n\nfolds = pd.read_csv(FOLD_PATH)\nif len(folds) != 2781:\n    raise RuntimeError("Frozen fold file row count mismatch")\nif sha256_bytes(FOLD_PATH)[:16] != FOLD_SHA16:\n    raise RuntimeError(f"Frozen fold sha mismatch: {sha256_bytes(FOLD_PATH)[:16]}")\nif sorted(folds["fold"].unique().tolist()) != [0, 1, 2, 3, 4]:\n    raise RuntimeError(f"Unexpected fold labels: {sorted(folds[\'fold\'].unique().tolist())}")\nif not folds["Id"].equals(train["Id"]):\n    raise RuntimeError("Frozen fold Id order does not align with train")\n\ny = train["Drafted"].to_numpy(dtype=int)\nif sorted(np.unique(y).tolist()) != [0, 1]:\n    raise RuntimeError("Target is not binary 0/1")\n\nfeatures = add_f2_features(train)\ndiagnostics = diagnostic_frame(train, features)\n\nm0_oof = pd.read_csv(M0_OOF_PATH)\nm1_oof = pd.read_csv(M1_OOF_PATH)\nvalidate_oof_frame(m0_oof, folds, "m0")\nvalidate_oof_frame(m1_oof, folds, "m1")\nm0_auc = auc_safe(m0_oof["y_true"].to_numpy(), m0_oof["y_pred_proba"].to_numpy())\nm1_auc = auc_safe(m1_oof["y_true"].to_numpy(), m1_oof["y_pred_proba"].to_numpy())\nif abs(m0_auc - M0_EXPECTED_AUC) > M0_TOL:\n    raise RuntimeError(f"M0 AUC mismatch: {m0_auc}")\n\nm0_fold_auc = fold_auc_table("m0", m0_oof)\nm1_fold_auc = fold_auc_table("m1", m1_oof)\n\nregistry = []\nif XGBClassifier is not None:\n    registry.append({\n        "model_key": "xgboost",\n        "factory": lambda: XGBClassifier(\n            n_estimators=300,\n            max_depth=6,\n            learning_rate=0.1,\n            subsample=1.0,\n            colsample_bytree=1.0,\n            reg_lambda=1.0,\n            random_state=42,\n            n_jobs=1,\n            tree_method="hist",\n            eval_metric="logloss",\n            verbosity=0,\n        ),\n        "fit_kwargs": {},\n        "dependency_status": "ok",\n    })\nelse:\n    registry.append({"model_key": "xgboost", "factory": None, "fit_kwargs": {}, "dependency_status": XGBOOST_IMPORT_ERROR})\n\nif LGBMClassifier is not None:\n    registry.append({\n        "model_key": "lightgbm",\n        "factory": lambda: LGBMClassifier(\n            n_estimators=300,\n            num_leaves=31,\n            learning_rate=0.1,\n            subsample=1.0,\n            colsample_bytree=1.0,\n            reg_lambda=1.0,\n            random_state=42,\n            n_jobs=1,\n            deterministic=True,\n            force_col_wise=True,\n            verbose=-1,\n        ),\n        "fit_kwargs": {},\n        "dependency_status": "ok",\n    })\nelse:\n    registry.append({"model_key": "lightgbm", "factory": None, "fit_kwargs": {}, "dependency_status": LIGHTGBM_IMPORT_ERROR})\n\ncatboost_gate_ok = (\n    CATBOOST_2B_AUTHORIZED\n    and CATBOOST_SCHOOL_EXCLUSION_RECONFIRMED\n    and CatBoostClassifier is not None\n    and "School" not in features.columns\n)\nif catboost_gate_ok:\n    registry.append({\n        "model_key": "catboost",\n        "factory": lambda: CatBoostClassifier(\n            iterations=300,\n            depth=6,\n            learning_rate=0.1,\n            random_seed=42,\n            thread_count=1,\n            allow_writing_files=False,\n            verbose=False,\n            loss_function="Logloss",\n        ),\n        "fit_kwargs": {"cat_features": []},\n        "dependency_status": "ok",\n    })\nelse:\n    registry.append({\n        "model_key": "catboost",\n        "factory": None,\n        "fit_kwargs": {},\n        "dependency_status": CATBOOST_IMPORT_ERROR or "catboost_gate_failed_or_not_authorized",\n    })\n\noof_frames: dict[str, pd.DataFrame] = {}\nfold_rows: list[dict] = []\nsummary_rows: list[dict] = []\nslice_rows: list[dict] = []\n\nfor entry in registry:\n    model_key = entry["model_key"]\n    if entry["factory"] is None:\n        summary_rows.append({\n            "model_key": model_key,\n            "run_status": "failed_or_skipped",\n            "oof_auc": np.nan,\n            "delta_vs_m0": np.nan,\n            "delta_vs_m1": np.nan,\n            "fold_auc_mean": np.nan,\n            "fold_auc_std": np.nan,\n            "same_sign_positive_folds_vs_m0": 0,\n            "same_sign_positive_folds_vs_m1": 0,\n            "slice_guard_triggered": False,\n            "classification": "failed_run",\n            "notes": entry["dependency_status"],\n        })\n        continue\n\n    oof = np.full(len(train), np.nan, dtype=float)\n    fold_aucs: dict[int, float] = {}\n    for fold in [0, 1, 2, 3, 4]:\n        train_mask = folds["fold"].to_numpy() != fold\n        valid_mask = folds["fold"].to_numpy() == fold\n        y_train = y[train_mask]\n        y_valid = y[valid_mask]\n        if len(np.unique(y_valid)) < 2:\n            raise RuntimeError(f"{model_key} fold {fold} validation target has one class")\n        preprocessor = make_preprocessor()\n        x_train = preprocessor.fit_transform(features.loc[train_mask, :])\n        x_valid = preprocessor.transform(features.loc[valid_mask, :])\n        if "School" in getattr(preprocessor, "feature_names_in_", []):\n            raise RuntimeError("School appeared in preprocessing input")\n        model = entry["factory"]()\n        model.fit(x_train, y_train, **entry["fit_kwargs"])\n        fold_proba = positive_class_proba(model, x_valid)\n        oof[valid_mask] = fold_proba\n        fold_auc = auc_safe(y_valid, fold_proba)\n        fold_aucs[fold] = fold_auc\n        fold_rows.append({\n            "model_key": model_key,\n            "fold": fold,\n            "auc": fold_auc,\n            "m0_auc": m0_fold_auc[fold],\n            "m1_auc": m1_fold_auc[fold],\n            "delta_vs_m0": fold_auc - m0_fold_auc[fold],\n            "delta_vs_m1": fold_auc - m1_fold_auc[fold],\n        })\n\n    if np.isnan(oof).any():\n        raise RuntimeError(f"{model_key} OOF has missing predictions")\n    if not np.isfinite(oof).all() or ((oof < 0.0) | (oof > 1.0)).any():\n        raise RuntimeError(f"{model_key} invalid OOF probabilities")\n\n    oof_frame = pd.DataFrame({\n        "Id": train["Id"],\n        "fold": folds["fold"],\n        "y_true": y,\n        "y_pred_proba": oof,\n    })\n    validate_oof_frame(oof_frame, folds, model_key)\n    oof_frames[model_key] = oof_frame\n\n    oof_auc = auc_safe(y, oof)\n    same_vs_m0 = int(sum(fold_aucs[f] - m0_fold_auc[f] > 0 for f in [0, 1, 2, 3, 4]))\n    same_vs_m1 = int(sum(fold_aucs[f] - m1_fold_auc[f] > 0 for f in [0, 1, 2, 3, 4]))\n\n    model_slice_guard = False\n    for slice_name in [\n        "Player_Type",\n        "Position_Type",\n        "Year",\n        "Age_missing",\n        "available_measurement_count",\n        "measurement_completeness_group",\n        "frequent_vs_rare_school_group",\n    ]:\n        for slice_value in sorted(diagnostics[slice_name].astype(str).unique()):\n            mask = diagnostics[slice_name].astype(str) == slice_value\n            n = int(mask.sum())\n            n_positive = int(y[mask].sum())\n            n_negative = int(n - n_positive)\n            status = "computed"\n            reason = ""\n            model_auc = np.nan\n            m0_slice_auc = np.nan\n            m1_slice_auc = np.nan\n            delta_vs_m0 = np.nan\n            delta_vs_m1 = np.nan\n            slice_guard = False\n            if n < SLICE_MIN_N:\n                status = "skipped_too_small"\n                reason = f"n<{SLICE_MIN_N}"\n            elif n_positive == 0 or n_negative == 0:\n                status = "skipped_one_class"\n                reason = "slice has one target class"\n            else:\n                model_auc = auc_safe(y[mask], oof[mask])\n                m0_slice_auc = auc_safe(y[mask], m0_oof.loc[mask, "y_pred_proba"].to_numpy())\n                m1_slice_auc = auc_safe(y[mask], m1_oof.loc[mask, "y_pred_proba"].to_numpy())\n                delta_vs_m0 = model_auc - m0_slice_auc\n                delta_vs_m1 = model_auc - m1_slice_auc\n                slice_guard = bool(delta_vs_m0 < SLICE_DEGRADATION_THRESHOLD)\n                if slice_guard:\n                    model_slice_guard = True\n            slice_rows.append({\n                "model_key": model_key,\n                "slice_name": slice_name,\n                "slice_value": slice_value,\n                "n": n,\n                "n_positive": n_positive,\n                "n_negative": n_negative,\n                "positive_rate": n_positive / n if n else np.nan,\n                "model_auc": model_auc,\n                "m0_auc": m0_slice_auc,\n                "m1_auc": m1_slice_auc,\n                "delta_vs_m0": delta_vs_m0,\n                "delta_vs_m1": delta_vs_m1,\n                "status": status,\n                "reason_if_skipped": reason,\n                "slice_guard_triggered": slice_guard,\n                "notes": "Age_missing=1 tracked explicitly" if slice_name == "Age_missing" and slice_value == "1" else "",\n            })\n\n    delta_m0 = oof_auc - m0_auc\n    delta_m1 = oof_auc - m1_auc\n    classification = classification_for(delta_m0, same_vs_m0, model_slice_guard)\n    summary_rows.append({\n        "model_key": model_key,\n        "run_status": "completed",\n        "oof_auc": oof_auc,\n        "delta_vs_m0": delta_m0,\n        "delta_vs_m1": delta_m1,\n        "fold_auc_mean": float(np.mean(list(fold_aucs.values()))),\n        "fold_auc_std": float(np.std(list(fold_aucs.values()), ddof=1)),\n        "same_sign_positive_folds_vs_m0": same_vs_m0,\n        "same_sign_positive_folds_vs_m1": same_vs_m1,\n        "slice_guard_triggered": model_slice_guard,\n        "classification": classification,\n        "notes": "No winner declared; classification is evidence only.",\n    })\n\nsummary_df = pd.DataFrame(summary_rows)\nfold_df = pd.DataFrame(fold_rows)\nslice_df = pd.DataFrame(slice_rows)\n\nfor model_key, frame in oof_frames.items():\n    safe_write_csv(REPO_ROOT / "outputs" / "oof" / f"{EXPERIMENT_ID}_{model_key}_oof_predictions.csv", frame)\nsafe_write_csv(MODEL_SUMMARY_PATH, summary_df)\nsafe_write_csv(FOLD_METRICS_PATH, fold_df)\nsafe_write_csv(SLICE_REPORT_PATH, slice_df)\n\ncandidate_rows = []\nfor _, row in summary_df.iterrows():\n    candidate_rows.append({\n        "experiment_id": EXPERIMENT_ID,\n        "model_key": row["model_key"],\n        "phase": "Phase 8 Wave 2",\n        "notebook_or_script": "notebooks/08c_phase8_wave2_external_gbdt_comparison.ipynb",\n        "git_commit_or_status": head,\n        "feature_block": "F2",\n        "model_family": row["model_key"],\n        "hpo_status": "not_run",\n        "cv_auc_mean": row["fold_auc_mean"],\n        "cv_auc_std": row["fold_auc_std"],\n        "oof_auc": row["oof_auc"],\n        "delta_vs_m0": row["delta_vs_m0"],\n        "delta_vs_m1": row["delta_vs_m1"],\n        "slice_metrics_available": True,\n        "leakage_checks_passed": row["classification"] != "failed_run",\n        "submission_created": False,\n        "submission_path": "",\n        "decision": row["classification"],\n        "notes": row["notes"],\n    })\nsafe_write_csv(CANDIDATE_LOG_PATH, pd.DataFrame(candidate_rows))\n\nage_rows = slice_df[(slice_df["slice_name"] == "Age_missing") & (slice_df["slice_value"] == "1")]\nreport_lines = [\n    "# Phase 8 Wave 2 External GBDT Validation Report",\n    "",\n    "## Executive Summary",\n    "",\n    "Phase 8 Wave 2 executed the authorized external GBDT registry only. The run compares XGBoost, LightGBM, and CatBoost if gated/importable against persisted M0 and M1 OOF comparators. It does not declare a final winner.",\n    "",\n    "## Scope Guardrails",\n    "",\n    "- Feature set: accepted F2 only.",\n    "- Excluded from feature matrices: Id, Drafted, School.",\n    "- School used only for diagnostic slices.",\n    "- No HPO, no early stopping, no eval_set, no submissions, no leaderboard use, no external data.",\n    "- M0 and M1 were loaded from persisted Wave 1 OOF artifacts and were not retrained.",\n    "- Phase 9, Phase 10, and Phase 11 remain locked.",\n    "",\n    "## Environment",\n    "",\n    f"- Python: `{platform.python_version()}`",\n    f"- numpy: `{np.__version__}`",\n    f"- pandas: `{pd.__version__}`",\n    f"- scikit-learn: imported through sklearn runtime",\n    f"- xgboost import error: `{XGBOOST_IMPORT_ERROR}`",\n    f"- lightgbm import error: `{LIGHTGBM_IMPORT_ERROR}`",\n    f"- catboost import error: `{CATBOOST_IMPORT_ERROR}`",\n    "",\n    "## Anchor Checks",\n    "",\n    f"- M0 OOF ROC-AUC: `{m0_auc}`",\n    f"- M1 OOF ROC-AUC: `{m1_auc}`",\n    f"- Frozen fold sha256[:16]: `{sha256_bytes(FOLD_PATH)[:16]}`",\n    "",\n    "## Model Summary",\n    "",\n    markdown_table(summary_df),\n    "",\n    "## Fold Metrics",\n    "",\n    markdown_table(fold_df) if not fold_df.empty else "No trained Wave 2 model fold metrics were produced.",\n    "",\n    "## Age_missing=1 Slice Tracking",\n    "",\n    markdown_table(age_rows) if not age_rows.empty else "No Age_missing=1 slice rows were available.",\n    "",\n    "## Leakage Checklist",\n    "",\n    "- All learned preprocessing was fit inside each training fold.",\n    "- Test data was not used for fitting, tuning, preprocessing, selection, or inference.",\n    "- No target encoding, feature selection, HPO, early stopping, eval_set, submissions, leaderboard use, or external data were used.",\n    "- CatBoost, when run, used the same pre-encoded numeric F2 matrix with `cat_features=[]`.",\n    "- `logs/experiment_log.csv` was checked byte-identical before and after the run.",\n    "",\n    "## Final Note",\n    "",\n    "Wave 2 produces classified evidence only. Candidate selection, acceptance, and commit remain gated on explicit project director authorization.",\n]\nsafe_write_text(VALIDATION_REPORT_PATH, "\\n".join(report_lines) + "\\n")\n\nmanifest_rows = []\nfor path in [\n    *[REPO_ROOT / "outputs" / "oof" / f"{EXPERIMENT_ID}_{model_key}_oof_predictions.csv" for model_key in oof_frames],\n    MODEL_SUMMARY_PATH,\n    FOLD_METRICS_PATH,\n    SLICE_REPORT_PATH,\n    DEPENDENCY_REPORT,\n    VALIDATION_REPORT_PATH,\n    CANDIDATE_LOG_PATH,\n    ENVIRONMENT_REPORT,\n]:\n    if path.exists():\n        rows = ""\n        try:\n            if path.suffix.lower() == ".csv":\n                rows = str(max(sum(1 for _ in path.open("r", encoding="utf-8")) - 1, 0))\n        except Exception:\n            rows = ""\n        manifest_rows.append({\n            "path": str(path.relative_to(REPO_ROOT)),\n            "sha256": sha256_bytes(path),\n            "rows": rows,\n            "bytes": path.stat().st_size,\n        })\nmanifest_df = pd.DataFrame(manifest_rows)\nsafe_write_csv(ARTIFACT_MANIFEST_PATH, manifest_df)\n\nif sha256_bytes(MAIN_LOG_PATH) != log_before:\n    raise RuntimeError("logs/experiment_log.csv changed during Wave 2 execution")\nif run_cmd(["git", "diff", "--name-only", "--", *FORBIDDEN_PATHS]).stdout.strip():\n    raise RuntimeError("Forbidden-path diff present after Wave 2 execution")\n\nprint(json.dumps({\n    "m0_auc": m0_auc,\n    "m1_auc": m1_auc,\n    "models": summary_df[["model_key", "classification", "oof_auc", "delta_vs_m0", "delta_vs_m1"]].to_dict(orient="records"),\n    "artifacts": [row["path"] for row in manifest_rows],\n}, indent=2))\n', encoding="utf-8")
print("Wrote temporary runner outside repo:", TEMP_RUNNER)


$ git rev-parse HEAD
98d8bb9bc0cc2cccd0c3722a9efebf56ab63021e
$ git diff --cached --name-only
$ git diff --check
$ git diff --name-only -- data/input notebooks/_official references outputs/submissions logs/experiment_log.csv .vscode/settings.json
Wrote temporary runner outside repo: C:\tmp\phase8_wave2_external_gbdt_v1_runner.py


## Execute Wave 2 Runner in the Separate Environment


In [2]:
result = run_cmd([str(ENV_PYTHON), str(TEMP_RUNNER)], check=False)
if result.returncode != 0:
    raise RuntimeError("Wave 2 comparison runner failed. See captured stderr/stdout above.")
print("Wave 2 comparison runner: PASS")


$ C:\tmp\reto_tokio_phase8_wave2_env\Scripts\python.exe C:\tmp\phase8_wave2_external_gbdt_v1_runner.py
{
  "m0_auc": 0.8116502602456482,
  "m1_auc": 0.8270821069632867,
  "models": [
    {
      "model_key": "xgboost",
      "classification": "no_qualifying_evidence",
      "oof_auc": 0.8113477083751576,
      "delta_vs_m0": -0.0003025518704906638,
      "delta_vs_m1": -0.015734398588129084
    },
    {
      "model_key": "lightgbm",
      "classification": "no_qualifying_evidence",
      "oof_auc": 0.8062204891415921,
      "delta_vs_m0": -0.00542977110405618,
      "delta_vs_m1": -0.0208616178216946
    },
    {
      "model_key": "catboost",
      "classification": "escalated",
      "oof_auc": 0.8202943968641223,
      "delta_vs_m0": 0.008644136618474074,
      "delta_vs_m1": -0.006787710099164346
    }
  ],
  "artifacts": [
    "outputs\\oof\\phase8_wave2_external_gbdt_v1_xgboost_oof_predictions.csv",
    "outputs\\oof\\phase8_wave2_external_gbdt_v1_lightgbm_oof_predictions.csv",


## Post-Run Guard Verification


In [3]:
run_cmd(["git", "diff", "--check"])
assert not run_cmd(["git", "diff", "--cached", "--name-only"]).stdout.strip(), "Staged files present after run"
assert not run_cmd(["git", "diff", "--name-only", "--", *FORBIDDEN_PATHS]).stdout.strip(), "Forbidden-path diff after run"
print("Post-run guards: PASS")


$ git diff --check
$ git diff --cached --name-only
$ git diff --name-only -- data/input notebooks/_official references outputs/submissions logs/experiment_log.csv .vscode/settings.json
Post-run guards: PASS
